In [1]:
import pandas as pd 
import numpy as np

In [2]:
df = pd.read_csv("D:/nassau_profitability_project/data/processed/nassau_cleaned.csv",parse_dates=["order_date",'ship_date'])
df.head()

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,country_region,city,state_province,postal_code,...,profit_diff_abs,shipping_days,factory,latitude,longitude,year,month,month_name,quarter,year_month
0,1,US-2021-103800-CHO-MIL-31000,2024-01-03,2026-06-30,Standard Class,103800,United States,Houston,Texas,77095,...,0.0,909,Wicked Choccy's,32.076176,-81.088371,2024,1,January,1,2024-01
1,2,US-2021-112326-CHO-TRI-54000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,0.0,909,Wicked Choccy's,32.076176,-81.088371,2024,1,January,1,2024-01
2,3,US-2021-112326-CHO-NUT-13000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,0.0,909,Lot's O' Nuts,32.881893,-111.768036,2024,1,January,1,2024-01
3,4,US-2021-112326-CHO-SCR-58000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,0.0,909,Lot's O' Nuts,32.881893,-111.768036,2024,1,January,1,2024-01
4,5,US-2021-141817-CHO-TRI-54000,2024-01-05,2026-07-05,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,...,0.0,912,Wicked Choccy's,32.076176,-81.088371,2024,1,January,1,2024-01


In [10]:
#Basic Sanity check
print(df.shape)
df.info()
df[["sales","cost", "gross_profit", "units"]].describe()

(10194, 30)
<class 'pandas.DataFrame'>
RangeIndex: 10194 entries, 0 to 10193
Data columns (total 30 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   row_id             10194 non-null  int64         
 1   order_id           10194 non-null  str           
 2   order_date         10194 non-null  datetime64[us]
 3   ship_date          10194 non-null  datetime64[us]
 4   ship_mode          10194 non-null  str           
 5   customer_id        10194 non-null  int64         
 6   country_region     10194 non-null  str           
 7   city               10194 non-null  str           
 8   state_province     10194 non-null  str           
 9   postal_code        10194 non-null  str           
 10  division           10194 non-null  str           
 11  region             10194 non-null  str           
 12  product_id         10194 non-null  str           
 13  product_name       10194 non-null  str           
 14  sales

,sales,cost,gross_profit,units
count,10194.000000,10194.000000,10194.000000,10194.000000
mean,13.908537,4.742087,9.166451,3.791838
std,11.341020,5.061647,6.643740,2.228317
min,1.250000,0.600000,0.250000,1.000000
25%,7.200000,2.400000,4.900000,2.000000
50%,10.800000,3.600000,7.470000,3.000000
75%,18.000000,5.700000,12.250000,5.000000
max,260.000000,130.000000,130.000000,14.000000


In [11]:
# Gross Margin %
df["gross_margin_pct"] = (df["gross_profit"] / df["sales"]) * 100

#profit per unit 
df["profit_per_unit"]= (df["gross_profit"]/df["units"])

# sales per unit (selling price )
df["sales_per_unit"]= (df["sales"]/df["units"])

#cost per units 
df["cost_per_unit"] = (df["cost"]/df["units"])

# Cost ratio (efficiency)
df["cost_ratio"] = df["cost"] / df["sales"]

# Markup %
df["markup_pct"] = (df["gross_profit"] / df["cost"]) * 100

In [13]:
df[["gross_margin_pct","profit_per_unit","cost_ratio"]].describe()

,gross_margin_pct,profit_per_unit,cost_ratio
count,10194.000000,10194.000000,10194.000000
mean,66.514045,2.412539,0.334860
std,6.721012,0.806007,0.067210
min,7.692308,0.250000,0.200000
25%,65.333333,2.400000,0.305556
50%,66.666667,2.450000,0.333333
75%,69.444444,2.490000,0.346667
max,80.000000,10.000000,0.923077


In [14]:
df[df["gross_margin_pct"] > 100]
df[df["gross_margin_pct"] < -100]

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,country_region,city,state_province,postal_code,...,month,month_name,quarter,year_month,gross_margin_pct,profit_per_unit,sales_per_unit,cost_per_unit,cost_ratio,markup_pct


In [ ]:
#create margin risk flag
def margin_flag(x):
    if x < 10:
        return "High Risk"
    elif x < 20:
        return "Medium Risk"
    else:
        return "Healthy"

df["margin_risk_flag"] = df["gross_margin_pct"].apply(margin_flag)

In [16]:
df["margin_risk_flag"].value_counts()

margin_risk_flag
Healthy      10098
High Risk       96
Name: count, dtype: int64

In [17]:
#create cost efficiency flag
def cost_flag(x):
    if x > 0.9:
        return "High Cost"
    elif x > 0.75:
        return "Moderate Cost"
    else:
        return "Efficient"

df["cost_efficiency_flag"] = df["cost_ratio"].apply(cost_flag)

In [18]:
#create profit category (ML)
df["profit_category"] = pd.qcut(
    df["gross_profit"],
    q=3,
    labels=["Low Profit", "Medium Profit", "High Profit"]
)

In [19]:
#create product segmentation
#step 1 
sales_median = df["sales"].median()
profit_median = df["gross_profit"].median()
margin_median = df["gross_margin_pct"].median()

In [20]:
#step 2(segmentation logic )
def product_segment(row):
    if row["sales"] >= sales_median and row["gross_margin_pct"] < margin_median:
        return "High Sales Low Margin"
    
    elif row["gross_profit"] >= profit_median and row["gross_margin_pct"] >= margin_median:
        return "Star Product"
    
    elif row["gross_margin_pct"] >= margin_median and row["sales"] < sales_median:
        return "High Margin Low Sales"
    
    else:
        return "Low Performer"

df["product_segment"] = df.apply(product_segment, axis=1)

In [21]:
df["product_segment"].value_counts()

product_segment
High Sales Low Margin    3028
Star Product             2778
Low Performer            2502
High Margin Low Sales    1886
Name: count, dtype: int64

In [22]:
#contribution metrices
total_sales = df["sales"].sum()
total_profit = df["gross_profit"].sum()

df["revenue_contribution_pct"] = (df["sales"] / total_sales) * 100
df["profit_contribution_pct"] = (df["gross_profit"] / total_profit) * 100

In [23]:
df.head()

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,country_region,city,state_province,postal_code,...,sales_per_unit,cost_per_unit,cost_ratio,markup_pct,margin_risk_flag,cost_efficiency_flag,profit_category,product_segment,revenue_contribution_pct,profit_contribution_pct
0,1,US-2021-103800-CHO-MIL-31000,2024-01-03,2026-06-30,Standard Class,103800,United States,Houston,Texas,77095,...,3.25,1.14,0.350769,185.087719,Healthy,Efficient,Low Profit,Low Performer,0.004584,0.004516
1,2,US-2021-112326-CHO-TRI-54000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,3.75,1.30,0.346667,188.461538,Healthy,Efficient,Low Profit,Low Performer,0.005290,0.005244
2,3,US-2021-112326-CHO-NUT-13000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,3.49,1.00,0.286533,249.000000,Healthy,Efficient,Medium Profit,Star Product,0.007384,0.007994
3,4,US-2021-112326-CHO-SCR-58000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,3.60,1.10,0.305556,227.272727,Healthy,Efficient,Medium Profit,Star Product,0.007617,0.008026
4,5,US-2021-141817-CHO-TRI-54000,2024-01-05,2026-07-05,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,...,3.75,1.30,0.346667,188.461538,Healthy,Efficient,Medium Profit,High Sales Low Margin,0.007935,0.007866


In [24]:
df.columns

Index(['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode',
       'customer_id', 'country_region', 'city', 'state_province',
       'postal_code', 'division', 'region', 'product_id', 'product_name',
       'sales', 'units', 'gross_profit', 'cost', 'calculated_profit',
       'profit_diff', 'profit_diff_abs', 'shipping_days', 'factory',
       'latitude', 'longitude', 'year', 'month', 'month_name', 'quarter',
       'year_month', 'gross_margin_pct', 'profit_per_unit', 'sales_per_unit',
       'cost_per_unit', 'cost_ratio', 'markup_pct', 'margin_risk_flag',
       'cost_efficiency_flag', 'profit_category', 'product_segment',
       'revenue_contribution_pct', 'profit_contribution_pct'],
      dtype='str')